# Anatomy of a `.rete` — build one from an rdflib graph

This notebook builds a **knowledge graph with a small ontology** in
[rdflib](https://rdflib.readthedocs.io), turns it into a **`.rete` file**,
and then dissects every part of that file: the **dictionary**, the **triple
indexes**, the **pyramid**, the **Dataset Card**, and the **embedded example
queries**. Everything runs in your browser tab (Pyodide + JupyterLite — no
server anywhere).

Project: <https://github.com/caviri/rete> · docs:
[Python API](https://caviri.github.io/rete/python.html) ·
[format spec](https://caviri.github.io/rete/SPEC.html)

In [ ]:
%pip install rete-graph rdflib pandas

import pandas as pd
import rdflib
import rete_graph as rete
print(f"rete-graph {rete.__version__} · rdflib {rdflib.__version__}")

## 1. A small KG + ontology, in rdflib

Eight bodies of the solar system, and — this is the part that makes the rest
of the notebook interesting — a tiny **ontology** on top: `Planet` and
`Moon` are both `rdfs:subClassOf` **`CelestialBody`**, and the `orbits`
property declares its domain and range. No instance is ever typed as a
`CelestialBody` directly; later, the reasoner will *infer* that they all are.

In [ ]:
from rdflib import RDF, RDFS, XSD, Literal, Namespace

SOL = Namespace("https://example.org/solar/")
kg = rdflib.Graph()

# --- the ontology (TBox) ---
for cls in ("CelestialBody", "Planet", "Moon"):
    kg.add((SOL[cls], RDF.type, RDFS.Class))
kg.add((SOL.Planet, RDFS.subClassOf, SOL.CelestialBody))
kg.add((SOL.Moon, RDFS.subClassOf, SOL.CelestialBody))
kg.add((SOL.orbits, RDFS.domain, SOL.CelestialBody))
kg.add((SOL.orbits, RDFS.range, SOL.CelestialBody))

# --- the data (ABox) ---
bodies = {
    "Sun":     (None,      None,   695_700),
    "Mercury": ("Planet", "Sun",     2_440),
    "Venus":   ("Planet", "Sun",     6_052),
    "Earth":   ("Planet", "Sun",     6_371),
    "Moon":    ("Moon",   "Earth",   1_737),
    "Mars":    ("Planet", "Sun",     3_390),
    "Phobos":  ("Moon",   "Mars",       11),
    "Deimos":  ("Moon",   "Mars",        6),
}
for name, (cls, parent, radius) in bodies.items():
    node = SOL[name]
    kg.add((node, RDFS.label, Literal(name, lang="en")))
    kg.add((node, SOL.radius_km, Literal(radius, datatype=XSD.integer)))
    if cls:
        kg.add((node, RDF.type, SOL[cls]))
    if parent:
        kg.add((node, SOL.orbits, SOL[parent]))

print(f"{len(kg)} triples in the rdflib graph — a taste, as Turtle:\n")
print("\n".join(kg.serialize(format="turtle").splitlines()[8:16]))

## 2. From rdflib to `.rete` — what the Builder assembles

`rete.Builder()` accepts the rdflib graph directly (anything with a
`.serialize()` method works). Configuration is **lazy** — each call just
records intent — and `run()` assembles the actual file, which is built from
four kinds of sections:

| Part | What it is | Why it exists |
|---|---|---|
| **Dictionary** | every distinct term (IRI, literal, bnode) stored **once**, sorted, in role-split sections | queries compare small integer ids, not strings; sorted order → prefix lookups & compression |
| **Triple indexes** | the id-triples in **three permutations** (SPO / POS / OSP) with zone maps | any triple pattern — `(s ? ?)`, `(? p ?)`, `(? ? o)` — becomes a *range scan*, which is also what makes lazy HTTP-range querying possible |
| **Pyramid** | a leveled *summary* of the graph — communities of nodes, rolled up level by level, plus the label index | "zoom out" views, the schema profile, and label autocomplete, all readable without touching the big indexes |
| **Dataset Card** | a JSON self-description in the metadata section | the file documents itself: title, license, counts, and runnable starter queries |

For the pyramid we choose `algo="types"`: **one community per `rdf:type`
class** — for an ontology-shaped graph like ours, the summary *is* the class
structure. (The default, `louvain`, clusters by connectivity instead.)

In [ ]:
builder = (
    rete.Builder()
    .add(kg)                                  # the rdflib graph, directly
    .card(
        title="Solar system micro-KG",
        description="Eight bodies + a tiny ontology (Planet/Moon ⊑ CelestialBody), "
                    "authored in rdflib inside a browser tab.",
        license="CC0-1.0",
        created="2026-07-17",
        # identity & provenance — any extra field goes straight into the card
        # JSON (the same curated fields the CLI's --card-file takes):
        version="1.0.0",
        creators=[{"name": "Ada Lovelace",
                   "orcid": "https://orcid.org/0000-0002-1825-0097"}],
        publisher={"name": "Example Observatory", "ror": "https://ror.org/02s376052"},
        derived_from=["https://example.org/solar-source"],
        cite_as="Lovelace, A. (2026). Solar system micro-KG.",
    )
    .example(
        "SELECT ?body ?parent WHERE { ?body <https://example.org/solar/orbits> ?parent }",
        title="Who orbits whom?",
        question="Which body orbits which other body?",
    )
    .example(
        "SELECT DISTINCT ?b WHERE { ?b a <https://example.org/solar/CelestialBody> }",
        title="All celestial bodies (needs reasoning!)",
        question="Which things are celestial bodies — including inferred ones?",
    )
    .pyramid(algo="types")                    # communities = classes
    .text_index()                             # opt-in full-text index
)

data = builder.run()
print(f"{len(data):,} bytes")
builder.stats

## 3. The dictionary — terms are stored once

Look at `stats` above: the file holds more **statements** than **terms**.
Every IRI like `sol:Sun` appears in many triples but is written once; each
triple is just three small integers pointing into the dictionary. That's the
same trick as HDT and column stores — and because the dictionary is sorted
and chunked, a remote client can fetch *only the chunks a query touches* and
run **prefix searches** against the label index without a full scan:

In [ ]:
g = builder.graph()
print(f"statements: {g.quads} · distinct terms: {g.terms}")
g.prefix_search("M")   # label index: everything labelled M… (Moon, Mars, Mercury)

## 4. The pyramid — the graph, summarized by class

`pyramidLevels` in the build stats came from the `types` algorithm: level 0
groups every node under its `rdf:type` class, and the **schema profile** is
exactly that quotient graph — which classes exist, and how they relate
through predicates. A client reads this in a couple of small range requests
*before* deciding what to query — it's the "map" you see first in the
[playground](https://caviri.github.io/rete/playground.html):

In [ ]:
s = g.schema()
display(pd.DataFrame(s["classes"], columns=["class", "instances"]))
pd.DataFrame(s["relations"], columns=["subject class", "predicate", "object class", "count"])

## 5. The Dataset Card — the file documents itself

The card travels **inside** the file's metadata section. We wrote the curated
fields — the catalog basics (`title`, `description`, `license`, `created`)
plus the **identity & provenance** fields (`version`, `creators` with ORCID
IRIs, `publisher` with a ROR IRI, `derived_from`, `cite_as`) — and the counts
and `format_version` were stamped automatically at build time. Every client
reads the same card — this notebook, the CLI (`rete card solar.rete`), the
playground's catalog view.

One honest limit: this builder **writes the card you supply — it cannot
derive one**. The `rete build` CLI additionally computes the enriched profile
(top predicates/classes, vocabularies, signals, the tiered starter-query
library) and the adjacent build-info record (timestamp, builder, parameters,
measured query costs); the JSON-LD / Croissant projections
(`rete card --format …`) are CLI-only too. To get those, rebuild the exported
file with the CLI.

In [ ]:
card = g.card()
{k: card[k] for k in ("title", "license", "version", "creators", "publisher",
                       "cite_as", "triple_count", "term_count", "format_version")}

## 6. Embedded example queries — a dataset that ships its own questions

The two `.example()` calls landed in the card's starter-query library. Anyone
who opens this file — today or in ten years, in any client — can list them
and run them as-is.

(Every query here matches the **default graph**, which is right for this
file — an rdflib `Graph` has no named graphs. If you build from N-Quads or an
rdflib `Dataset`, a bare pattern correctly returns *nothing* for statements
living in named graphs: scope it with `GRAPH ?g { … }`, or query it in the
playground with the opt-in ⛁ All graphs toggle — the non-standard
[union default graph](https://caviri.github.io/rete/sparql.html#union-default-graph)
mode.)

In [ ]:
examples = g.examples()
display(pd.DataFrame(examples)[["title", "question"]])
g.query_df(examples[0]["sparql"])   # "Who orbits whom?", straight from the file

## 7. The ontology at work — reasoning by query rewriting

Remember: nothing is *asserted* to be a `CelestialBody`. A plain query finds
nothing — but with `reason=True` the engine rewrites the query through the
ontology (`Planet ⊑ CelestialBody`, `Moon ⊑ CelestialBody`, plus the
`orbits` domain/range), computing OWL 2 QL entailment **without
materializing a single new triple**. That's why it works over lazy remote
files too:

In [ ]:
q = examples[1]["sparql"]           # "All celestial bodies" from the card
print("asserted only:", len(g.query(q)), "rows")
print("with OWL 2 QL reasoning:")
g.query_df(q, reason=True)          # planets, moons — and the Sun, via orbits' range

## 8. Export — one immutable file, queryable anywhere

`export()` writes the finished file (here into the notebook's in-browser
filesystem — check the file browser on the left; you can download it). Host
it on anything that serves HTTP `Range` and it is queryable in place from
Python, JavaScript, the CLI, and the playground — carrying its card, its
examples, and its ontology with it. For datasets beyond a few million
triples, graduate to the streaming
[`rete build` CLI](https://caviri.github.io/rete/cli.html).

In [ ]:
path = builder.export("solar.rete")
reopened = rete.open(path)
print(path, "·", reopened.quads, "triples · content hash", reopened.content_hash())

---

**Where next:** the [guided tour notebook](rete-graph.ipynb) (remote graphs,
lazy range reads) · [Python build tutorial](https://caviri.github.io/rete/python-build-tutorial.html) ·
[Dataset Cards](https://caviri.github.io/rete/dataset-cards.html) ·
[semantic zoom / pyramid](https://caviri.github.io/rete/semantic-zoom.html) ·
[reasoning](https://caviri.github.io/rete/reasoning.html)

Source & issues: **<https://github.com/caviri/rete>**

© 2026 Carlos Vivar Ríos — released under the
[Apache License 2.0](https://github.com/caviri/rete/blob/main/LICENSE).